## 0. Подготовка

In [1]:
# импорты
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import pyarrow


In [2]:
# отключаем экспоненциальное отображение чисел в pandas и numpy
pd.set_option('display.float_format', '{:.2f}'.format)
np.set_printoptions(suppress=True)
# Снимаем ограничение на число отображаемых столбцов в pandas
pd.set_option('display.max_columns', None)      # показывать все столбцы
pd.set_option('display.width', None)            # не ограничивать ширину вывода
pd.set_option('display.max_colwidth', None)     # не ограничивать ширину столбца

Dataset состоит из 3 файлов:
- credit_card_transactions-ibm_v2.csv - представляет из себя набор транзакий, совершенных с использованием пластиковых карт (эти данные доступны как банку, выпустившему карту (банку-эмитетнту), так и банку, обслуживающему платёжный терминал (банк-эквайер))
- sd254_cards.csv - характеристики каждой из карт, представленных в дата-сете (частично эти данные доступны и банку-эквайеру, в полном объёме банку-эмитенту)
- sd254_users.csv - данные о клиентах (у одного клиента может быть несколько карт) - доступны только банку-эмитенту
Система борьбы с мошенничеством действует как у банка-эмитента, так и у банка-эквайера. Мы будем разрабатывать систему с точки зрения банка-эмитента - т.е. банка, имеющего полный доступ к данным.
Дата-сет построен на предположении, что все транзакции, помеченные как мошеннические, действительно являются таковыми, а те, которые не помечены, являются легитимными.
Задача - построить модель предсказания того, является ли транзакция мошеннической или нет.
Дополнительная информация: в целях анонимизации датасета реальные имена пользователей заменены на номера. Также названия продавцов закодированы кодом. Ключ для расшифровки кода авторами датасета не предоставлен, но заявлено, что этот код уникален для каждого мерчанта и не имеет никакого значения, кроме как для идентификации (т.е. в нём не нужно искать зависимости)

In [3]:
# загрузка данных о транзакциях
transactions = pd.read_csv('data\credit_card_transactions-ibm_v2.csv')
transactions.head()

<>:2: SyntaxWarning: invalid escape sequence '\c'
<>:2: SyntaxWarning: invalid escape sequence '\c'
C:\Users\dmytr\AppData\Local\Temp\ipykernel_3884\3421166226.py:2: SyntaxWarning: invalid escape sequence '\c'
  transactions = pd.read_csv('data\credit_card_transactions-ibm_v2.csv')


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.00,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.00,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.00,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.00,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.00,5912,NaN,No


In [4]:
transactions.tail()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
24386895,1999,1,2020,2,27,22:23,$-54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.00,5541,NaN,No
24386896,1999,1,2020,2,27,22:24,$54.00,Chip Transaction,-5162038175624867091,Merrimack,NH,3054.00,5541,NaN,No
24386897,1999,1,2020,2,28,07:43,$59.15,Chip Transaction,2500998799892805156,Merrimack,NH,3054.00,4121,NaN,No
24386898,1999,1,2020,2,28,20:10,$43.12,Chip Transaction,2500998799892805156,Merrimack,NH,3054.00,4121,NaN,No
24386899,1999,1,2020,2,28,23:10,$45.13,Chip Transaction,4751695835751691036,Merrimack,NH,3054.00,5814,NaN,No


In [5]:
# Загрузка данных о картах
cards_data = pd.read_csv('data\sd254_cards.csv')
cards_data.head(10)


<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\dmytr\AppData\Local\Temp\ipykernel_3884\367480346.py:2: SyntaxWarning: invalid escape sequence '\s'
  cards_data = pd.read_csv('data\sd254_cards.csv')


,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,1,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,0,2,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,0,3,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,0,4,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No
5,1,0,Visa,Credit,4404898874682993,09/2003,736,YES,1,$27500,09/2003,2012,No
6,1,1,Visa,Debit,4001482973848631,07/2022,972,YES,2,$28508,02/2011,2011,No
7,1,2,Mastercard,Debit,5627220683410948,06/2022,48,YES,2,$9022,07/2003,2015,No
8,1,3,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,YES,2,$54,06/2010,2015,No
9,1,4,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,YES,1,$99,07/2006,2012,No


In [6]:
# Загрузка данных о клиентах
users_data = pd.read_csv('data/sd254_users.csv')
users_data.head()

,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1


## 1. Первичная обработка данных

### 1.1 Объеденяем таблицы


In [7]:
transactions.shape, cards_data.shape, users_data.shape

((24386900, 15), (6146, 13), (2000, 18))

In [8]:
# В transactions есть столбец 'User', а в users_data индекс — это идентификатор пользователя
# Добавляем к каждой строке информацию о карте, по которой была совершена транзакция
data = (
    transactions
    .merge(users_data, left_on='User', right_index=True, how='left')
    .merge(cards_data, left_on=['User', 'Card'], right_on=['User', 'CARD INDEX'], how='left')
)
data.shape

(24386900, 45)

In [9]:
# Проведем проверку правильности объединения данных. Для этого проверим, что нет пользователей, у которых 
# максимальный индекс карты (Card) превышает количество карт, указанных в Num Credit Cards.
# Для каждого пользователя находим максимальный индекс карты
max_card_index = data.groupby('User')['Card'].max()

# Получаем колонку Num Credit Cards для каждого пользователя (предполагается, что она одинакова для всех строк пользователя)
num_cards = data.groupby('User')['Num Credit Cards'].first()

# Проверяем условие: max(Card) <= Num Credit Cards - 1 (так как индексация карт начинается с 0)
check = (max_card_index <= (num_cards - 1))

# Выводим пользователей, у которых условие не выполняется
invalid_users = check[~check].index.tolist()
if invalid_users:
    print("Пользователи с несоответствием:", invalid_users)
else:
    print("Все пользователи имеют корректные индексы карт.")

Все пользователи имеют корректные индексы карт.


In [10]:
# Данные занимают большой объём оперативной памяти, удалим все ненужные объекты, чтобы освободить память для дальнейшей работы
import gc
del transactions, cards_data, users_data, max_card_index, num_cards, check, invalid_users
gc.collect()

0

### 1.2 Первичный анализ данных и их самые базовые преобразование

In [11]:
# В этом разделе мы смотрим типы данных, основные их характеристики и выполняем самые простые преобразования, направленные на оптимизацию памяти 
# и удобство работы с данными. Тут не будет feature engineering, кроме совершенно очевидных случаев, а также подробного анализа.
data.head()

,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.00,5300,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.00,5651,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.00,5912,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No


In [12]:
# Переименуем некоторы колонки для упроста дальнейшей работы
data = data.rename(columns={
    'Use Chip': 'Use_Chip',
    'Merchant Name': 'Merchant_Name',
    'Merchant City': 'Merchant_City',
    'Merchant State': 'Merchant_State',
    'Zip': 'Merchant_Zip',
    'Errors?': 'Errors',
    'Current Age': 'Current_Age',
    'Retirement Age': 'Retirement_Age',
    'Birth Year': 'Birth_Year',
    'Birth Month': 'Birth_Month',
    'Address': 'User_Address',
    'Apartment': 'User_Apartment',
    'City': 'User_City',
    'State': 'User_State',
    'Zipcode': 'User_Zip',
    'Latitude': 'User_Latitude',
    'Longitude': 'User_Longitude',
    'Per Capita Income - Zipcode': 'Per_Capita_Income_Zipcode',
    'Yearly Income - Person': 'Yearly_Income',
    'Total Debt': 'Total_Debt',
    'FICO Score': 'FICO',
    'Num Credit Cards': 'Num_Credit_Cards',
    'CARD INDEX': 'Card_Index',
    'Card Brand': 'Card_Brand',
    'Card Type': 'Card_Type',
    'Card Number': 'Card_Number',
    'Has Chip': 'Has_Chip',
    'Cards Issued': 'Cards_Issued',
    'Credit Limit': 'Credit_Limit',
    'Acct Open Date': 'Account_Open_Date',
    'Year PIN last Changed': "PIN_Last_Changed_Year",
    'Card on Dark Web': 'Card_on_Dark_Web',
    'Is Fraud?': 'Fraud'
})

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 45 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   User                       int64  
 1   Card                       int64  
 2   Year                       int64  
 3   Month                      int64  
 4   Day                        int64  
 5   Time                       object 
 6   Amount                     object 
 7   Use_Chip                   object 
 8   Merchant_Name              int64  
 9   Merchant_City              object 
 10  Merchant_State             object 
 11  Merchant_Zip               float64
 12  MCC                        int64  
 13  Errors                     object 
 14  Fraud                      object 
 15  Person                     object 
 16  Current_Age                int64  
 17  Retirement_Age             int64  
 18  Birth_Year                 int64  
 19  Birth_Month                int64  
 20  

Датасет очень большой, нам необходимо оптимизировать типы данных что бы обеспечить экономию места.

In [14]:
data.describe(include='all')

,User,Card,Year,Month,Day,Time,Amount,Use_Chip,Merchant_Name,Merchant_City,Merchant_State,Merchant_Zip,MCC,Errors,Fraud,Person,Current_Age,Retirement_Age,Birth_Year,Birth_Month,Gender,User_Address,User_Apartment,User_City,User_State,User_Zip,User_Latitude,User_Longitude,Per_Capita_Income_Zipcode,Yearly_Income,Total_Debt,FICO,Num_Credit_Cards,Card_Index,Card_Brand,Card_Type,Card_Number,Expires,CVV,Has_Chip,Cards_Issued,Credit_Limit,Account_Open_Date,PIN_Last_Changed_Year,Card_on_Dark_Web
count,24386900.00,24386900.00,24386900.00,24386900.00,24386900.00,24386900,24386900,24386900,24386900.00,24386900,21666079,21508765.00,24386900.00,388431,24386900,24386900,24386900.00,24386900.00,24386900.00,24386900.00,24386900,24386900,6624201.00,24386900,24386900,24386900.00,24386900.00,24386900.00,24386900,24386900,24386900,24386900.00,24386900.00,24386900.00,24386900,24386900,24386900.00,24386900,24386900.00,24386900,24386900.00,24386900,24386900,24386900.00,24386900
unique,NaN,NaN,NaN,NaN,NaN,1440,98953,3,NaN,13429,223,NaN,NaN,23,2,1993,NaN,NaN,NaN,NaN,2,1999,NaN,1286,51,NaN,NaN,NaN,1754,1948,1880,NaN,NaN,NaN,4,3,NaN,259,NaN,2,NaN,3653,303,NaN,1
top,NaN,NaN,NaN,NaN,NaN,12:31,$80.00,Swipe Transaction,NaN,ONLINE,CA,NaN,NaN,Insufficient Balance,No,Beckett Gonzalez,NaN,NaN,NaN,NaN,Female,702 Elm Drive,NaN,Houston,CA,NaN,NaN,NaN,$0,$34496,$0,NaN,NaN,NaN,Mastercard,Debit,NaN,02/2020,NaN,YES,NaN,$10600,02/2010,NaN,No
freq,NaN,NaN,NaN,NaN,NaN,30604,250984,15386082,NaN,2720821,2591830,NaN,NaN,242783,24357143,82355,NaN,NaN,NaN,NaN,12530577,82355,NaN,297299,2961905,NaN,NaN,NaN,135310,82355,1184490,NaN,NaN,NaN,13092130,15059468,NaN,737007,NaN,21910227,NaN,149176,300734,NaN,24386900
mean,1001.02,1.35,2011.96,6.53,15.72,NaN,NaN,NaN,-476922962771995648.00,NaN,NaN,50956.44,5561.17,NaN,NaN,NaN,53.88,66.38,1965.30,6.57,NaN,NaN,635.65,NaN,NaN,50206.03,37.34,-91.42,NaN,NaN,NaN,712.41,3.68,1.35,NaN,NaN,4822467873935394.00,NaN,491.25,NaN,1.53,NaN,NaN,2011.23,NaN
std,569.46,1.41,5.11,3.47,8.79,NaN,NaN,NaN,4758939870684009472.00,NaN,NaN,29397.07,879.32,NaN,NaN,NaN,15.84,3.63,15.83,3.58,NaN,NaN,1782.30,NaN,NaN,29485.66,5.15,16.29,NaN,NaN,NaN,67.03,1.62,1.41,NaN,NaN,1328106849335672.25,NaN,288.44,NaN,0.52,NaN,NaN,3.06,NaN
min,0.00,0.00,1991.00,1.00,1.00,NaN,NaN,NaN,-9222899435637403648.00,NaN,NaN,501.00,1711.00,NaN,NaN,NaN,18.00,50.00,1918.00,1.00,NaN,NaN,1.00,NaN,NaN,1060.00,20.88,-159.41,NaN,NaN,NaN,480.00,1.00,0.00,NaN,NaN,300105541992311.00,NaN,0.00,NaN,1.00,NaN,NaN,2002.00,NaN
25%,510.00,0.00,2008.00,3.00,8.00,NaN,NaN,NaN,-4500542936415012352.00,NaN,NaN,28374.00,5300.00,NaN,NaN,NaN,42.00,65.00,1956.00,3.00,NaN,NaN,5.00,NaN,NaN,28124.00,33.83,-97.34,NaN,NaN,NaN,683.00,3.00,0.00,NaN,NaN,4498188620007875.00,NaN,242.00,NaN,1.00,NaN,NaN,2009.00,NaN
50%,1006.00,1.00,2013.00,7.00,16.00,NaN,NaN,NaN,-794676495118551552.00,NaN,NaN,46742.00,5499.00,NaN,NaN,NaN,52.00,66.00,1968.00,7.00,NaN,NaN,10.00,NaN,NaN,46124.00,38.35,-86.28,NaN,NaN,NaN,714.00,4.00,1.00,NaN,NaN,5118758087775711.00,NaN,491.00,NaN,2.00,NaN,NaN,2011.00,NaN
75%,1477.00,2.00,2016.00,10.00,23.00,NaN,NaN,NaN,3189517333335617024.00,NaN,NaN,77564.00,5812.00,NaN,NaN,NaN,63.00,68.00,1977.00,10.00,NaN,NaN,94.00,NaN,NaN,77095.00,41.05,-80.13,NaN,NaN,NaN,755.00,5.00,2.00,NaN,NaN,5580555303942267.00,NaN,738.00,NaN,2.00,NaN,NaN,2013.00,NaN


In [15]:
# Смотрим число уникальных значений
data.nunique().sort_values()
# Понимаем, что необходимо анализировать каждый столбец отдельно, кроме того есть кандидаты на удаление и на преобразование

Card_on_Dark_Web                  1
Fraud                             2
Has_Chip                          2
Gender                            2
Card_Type                         3
Cards_Issued                      3
Use_Chip                          3
Card_Brand                        4
Card_Index                        9
Num_Credit_Cards                  9
Card                              9
Birth_Month                      12
Month                            12
PIN_Last_Changed_Year            19
Errors                           23
Retirement_Age                   29
Year                             30
Day                              31
User_State                       51
Birth_Year                       80
Current_Age                      80
MCC                             109
User_Apartment                  199
Merchant_State                  223
Expires                         259
Account_Open_Date               303
FICO                            321
User_Latitude               

In [16]:
data.isna().sum()

User                                0
Card                                0
Year                                0
Month                               0
Day                                 0
Time                                0
Amount                              0
Use_Chip                            0
Merchant_Name                       0
Merchant_City                       0
Merchant_State                2720821
Merchant_Zip                  2878135
MCC                                 0
Errors                       23998469
Fraud                               0
Person                              0
Current_Age                         0
Retirement_Age                      0
Birth_Year                          0
Birth_Month                         0
Gender                              0
User_Address                        0
User_Apartment               17762699
User_City                           0
User_State                          0
User_Zip                            0
User_Latitud

In [17]:
# Меняем типы данных для оптимизации памяти
data = data.astype({
    'User': 'int16',
    'Card': 'int8',
    'Year': 'int16',
    'Month': 'int8',
    'Day': 'int8',
    # 'Merchant_Zip': 'int32', # в данных есть пропуски, поэтому не можем привести к int32
    'MCC': 'category',
    'Current_Age': 'int8',
    'Retirement_Age': 'int8',
    'Birth_Year': 'int16',
    'Birth_Month': 'int8',
    # 'User_Apartment': 'int16', # в данных есть пропуски, поэтому не можем привести к int16
    'User_Zip': 'int32',
    'User_Latitude': 'float32',
    'User_Longitude': 'float32',
    'FICO': 'int16',
    'Num_Credit_Cards': 'int8',
    'Card_Index': 'int8',
    'CVV': 'int16',
    'Cards_Issued': 'int8',
})
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 45 columns):
 #   Column                     Dtype   
---  ------                     -----   
 0   User                       int16   
 1   Card                       int8    
 2   Year                       int16   
 3   Month                      int8    
 4   Day                        int8    
 5   Time                       object  
 6   Amount                     object  
 7   Use_Chip                   object  
 8   Merchant_Name              int64   
 9   Merchant_City              object  
 10  Merchant_State             object  
 11  Merchant_Zip               float64 
 12  MCC                        category
 13  Errors                     object  
 14  Fraud                      object  
 15  Person                     object  
 16  Current_Age                int8    
 17  Retirement_Age             int8    
 18  Birth_Year                 int16   
 19  Birth_Month        

In [18]:
data.head()

,User,Card,Year,Month,Day,Time,Amount,Use_Chip,Merchant_Name,Merchant_City,Merchant_State,Merchant_Zip,MCC,Errors,Fraud,Person,Current_Age,Retirement_Age,Birth_Year,Birth_Month,Gender,User_Address,User_Apartment,User_City,User_State,User_Zip,User_Latitude,User_Longitude,Per_Capita_Income_Zipcode,Yearly_Income,Total_Debt,FICO,Num_Credit_Cards,Card_Index,Card_Brand,Card_Type,Card_Number,Expires,CVV,Has_Chip,Cards_Issued,Credit_Limit,Account_Open_Date,PIN_Last_Changed_Year,Card_on_Dark_Web
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.00,5300,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.00,5651,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.00,5912,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No


In [19]:
# Переводим дату и время транзакции в единый формат datetime
data['Timestamp'] = pd.to_datetime(
    data['Year'].astype(str) + '-' +
    data['Month'].astype(str).str.zfill(2) + '-' +
    data['Day'].astype(str).str.zfill(2) + ' ' +
    data['Time'],
    format='%Y-%m-%d %H:%M'
)
# Перемещаем 'Timestamp' на третью позицию для удобства работы с данными
cols = list(data.columns)
cols.remove('Timestamp')
cols.insert(2, 'Timestamp')  # 2 — это третья позиция
data = data[cols]
# Удаляем теперь уже ненужные столбцы с датой и временем
data.drop(columns=['Year', 'Month', 'Day', 'Time'], inplace=True)

In [20]:
earliest = data['Timestamp'].min()
print("Самый ранний timestamp:", earliest)
latest = data['Timestamp'].max()
print("Самый последний timestamp:", latest)

Самый ранний timestamp: 1991-01-02 07:10:00
Самый последний timestamp: 2020-02-28 23:58:00


In [21]:
# Преобразуем столбец 'Amount' из строкового формата в числовой, удалив символ '$' и преобразовав тип данных в float
data["Amount"]=data["Amount"].str.replace("$","").astype('float32')
data['Amount'].describe()

count   24386900.00
mean          43.63
std           82.02
min         -500.00
25%            9.20
50%           30.14
75%           65.06
max        12390.50
Name: Amount, dtype: float64

In [22]:
# Авторы датасета сообщили о наличии ошибки - в ряде случаев при покупке в он-лайн магазине система
# фиксирует использование чипа, хотя на самом деле он не использовался. Поэтому проверим, сколько таких случаев в данных.
count = ((data['Merchant_City'] == 'ONLINE') & (data['Use_Chip'] == 'Chip Transaction')).sum()
print(f"Таких строк: {count}")

Таких строк: 7601


In [23]:
# Смотрим формат данных в столбце 'Use_Chip' и количество уникальных значений
data['Use_Chip'].value_counts()


Use_Chip
Swipe Transaction     15386082
Chip Transaction       6287598
Online Transaction     2713220
Name: count, dtype: int64

In [24]:
# Исправляем ошибку в данных, меняя значение 'Use_Chip' на 'Online Transaction' для строк, 
# где 'Merchant_City' равно 'ONLINE' и 'Use_Chip' равно 'Chip Transaction'
mask = (data['Merchant_City'] == 'ONLINE') & (data['Use_Chip'] == 'Chip Transaction')
data.loc[mask, 'Use_Chip'] = 'Online Transaction'

In [25]:
# Переводим столбец Use_Chip в категориальный тип данных для оптимизации памяти
data['Use_Chip'] = data['Use_Chip'].astype('category')

In [26]:
# Merchant_Name закодированы. Сам код нам ничего не даёт т.к. мы не имеем к нему ключа. Необходимо проверить, сколько всего 
# уникальных значений в этом столбце, чтобы понять, стоит ли его удалять либо кодировать
data['Merchant_Name'].nunique()

100343

In [27]:
# Кодируем имя мерчанта в идентификатор и преобразуем в категориальный тип данных для оптимизации памяти
data['Merchant_ID'] = data['Merchant_Name'].astype('category').cat.codes
# Перемещаем 'Merchant_ID' для удобства
cols = list(data.columns)
cols.remove('Merchant_ID')
cols.insert(5, 'Merchant_ID')  # 5 — это шестая позиция
data = data[cols]
data.drop(columns=['Merchant_Name'], inplace=True)
data['Merchant_ID'] = data['Merchant_ID'].astype('category')

In [28]:
# Смотрим состав городов, в которых были совершены транзакции, и количество уникальных городов
print(data['Merchant_City'].value_counts())
print(f"Количество уникальных городов: {data['Merchant_City'].nunique()}")

Merchant_City
ONLINE          2720821
Houston          246036
Los Angeles      180496
Miami            178653
Brooklyn         155425
                 ...   
Skellytown            1
Port Haywood          1
Island                1
Bluejacket            1
Geary                 1
Name: count, Length: 13429, dtype: int64
Количество уникальных городов: 13429


In [29]:
# Смотрим что имеется ввиду под states и сколько уникальных значений в этом столбце
print(data['Merchant_State'].value_counts())
print(f"Количество уникальных штатов: {data['Merchant_State'].nunique()}")

Merchant_State
CA                                  2591830
TX                                  1793298
FL                                  1458699
NY                                  1446864
OH                                   895970
                                     ...   
Togo                                      2
Democratic Republic of the Congo          2
Kiribati                                  1
Botswana                                  1
Paraguay                                  1
Name: count, Length: 223, dtype: int64
Количество уникальных штатов: 223


In [30]:
# Видим, что в датасете штаты как часть США и страны находятся в общем списке.
# На данный момент просто переводим в категориальный тип данных для оптимизации памяти, а дальше будем смотреть, что с этим делать
data['Merchant_City'] = data['Merchant_City'].astype('category')
data['Merchant_State'] = data['Merchant_State'].astype('category')

In [31]:
# Смотрим что у нас с ZIP-кодами мерчантов
print(data['Merchant_Zip'].value_counts())
print(f"Количество уникальных ZIP-кодов: {data['Merchant_Zip'].nunique()}")

Merchant_Zip
98516.00    55679
43830.00    48815
55024.00    44571
95076.00    43656
94606.00    43512
            ...  
49310.00        1
57456.00        1
56447.00        1
56257.00        1
56556.00        1
Name: count, Length: 27321, dtype: int64
Количество уникальных ZIP-кодов: 27321


In [32]:
# Пока тоже переводим в категориальный тип данных для оптимизации памяти, а дальше будем смотреть, что с этим делать
data['Merchant_Zip'] = data['Merchant_Zip'].astype('category')

In [33]:
# Смотрим информацию о столбцце Errors и количество уникальных значений в нём
print(data['Errors'].value_counts())
print(f"Количество уникальных ошибок: {data['Errors'].nunique()}")

Errors
Insufficient Balance                                   242783
Bad PIN                                                 58918
Technical Glitch                                        48157
Bad Card Number                                         13321
Bad CVV                                                 10740
Bad Expiration                                          10716
Bad Zipcode                                              2079
Bad PIN,Insufficient Balance                              581
Insufficient Balance,Technical Glitch                     457
Bad PIN,Technical Glitch                                  128
Bad Card Number,Insufficient Balance                      122
Bad CVV,Insufficient Balance                               89
Bad Expiration,Insufficient Balance                        78
Bad Card Number,Bad CVV                                    60
Bad Card Number,Bad Expiration                             54
Bad Expiration,Bad CVV                                     47
B

Делаем два важных вывода: нет ошибок, связанных со сработками анти-фрода - т.е. данный признак не будет вызывать утечку данных. Если бы были такие признаки, как "отклонено как мошенничество" то признак стал бы утечкой и его пришлось бы удалить. Есть также проблема - в некоторых случаях указаны сразу несколько ошибок, что создаст проблему для кодирования. Кодировать, скорее всего, будем через One_Hot или вообще заменим на флаг "произошла ошибка".

In [34]:
data.head()

,User,Card,Timestamp,Amount,Use_Chip,Merchant_ID,Merchant_City,Merchant_State,Merchant_Zip,MCC,Errors,Fraud,Person,Current_Age,Retirement_Age,Birth_Year,Birth_Month,Gender,User_Address,User_Apartment,User_City,User_State,User_Zip,User_Latitude,User_Longitude,Per_Capita_Income_Zipcode,Yearly_Income,Total_Debt,FICO,Num_Credit_Cards,Card_Index,Card_Brand,Card_Type,Card_Number,Expires,CVV,Has_Chip,Cards_Issued,Credit_Limit,Account_Open_Date,PIN_Last_Changed_Year,Card_on_Dark_Web
0,0,0,2002-09-01 06:21:00,134.09,Swipe Transaction,69374,La Verne,CA,91750.00,5300,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,0,2002-09-01 06:42:00,38.48,Swipe Transaction,46284,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2,0,0,2002-09-02 06:22:00,120.34,Swipe Transaction,46284,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
3,0,0,2002-09-02 17:45:00,128.95,Swipe Transaction,68751,Monterey Park,CA,91754.00,5651,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
4,0,0,2002-09-03 06:23:00,104.71,Swipe Transaction,81833,La Verne,CA,91750.00,5912,NaN,No,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No


По условиям датасета текущий возвраст не обновляется с течением времени - это запись из базы данных на момент создания датасета, а не на момент транзакции. Значит, нам необходимо будет в будущем пересчитать эти признаки

In [35]:
# Работаем с полом клиентов
print(data['Gender'].value_counts())

Gender
Female    12530577
Male      11856323
Name: count, dtype: int64


In [36]:
# Кодируем пол клиента как число, где 1 - мужчина, 0 - женщина
data['Gender'] = data['Gender'].map({'Male': 1, 'Female': 0}).astype('int8')

In [37]:
# Смотрим ZIP-коды клиентов
print(data['User_Zip'].value_counts())

User_Zip
95076    82355
10463    81689
43830    80749
98516    78861
98021    76427
         ...  
43211       26
57783       25
67467       21
17315       20
40071       15
Name: count, Length: 1815, dtype: int64


In [38]:
# Кодируем ZIP-коды клиентов в категориальный формат
data['User_Zip'] = data['User_Zip'].astype('category')

In [39]:
# Данные по доходам и долгам сейчас представителены в виде строк, необходимо удалить символы и привести к числовому формату для удобства работы с ними
data["Yearly_Income"]=data["Yearly_Income"].str.replace("$","").astype('float32')
data["Per_Capita_Income_Zipcode"]=data["Per_Capita_Income_Zipcode"].str.replace("$","").astype('float32')
data['Total_Debt']=data['Total_Debt'].str.replace("$","").astype('float32')


In [40]:
# Бренды карт и типы карт
print(data['Card_Brand'].value_counts())
print(data['Card_Type'].value_counts())

Card_Brand
Mastercard    13092130
Visa           8975874
Amex           1602563
Discover        716333
Name: count, dtype: int64
Card_Type
Debit              15059468
Credit              7673643
Debit (Prepaid)     1653789
Name: count, dtype: int64


In [41]:
# Меняем тип данных на категориальный
data['Card_Brand'] = data['Card_Brand'].astype('category')
data['Card_Type'] = data['Card_Type'].astype('category')

In [42]:
# Преобразуем дату окончания действия карты в формат datetime, добавляя последний день месяца для корректного отображения
data["Expires"] = pd.to_datetime(data["Expires"], format="%m/%Y") + pd.offsets.MonthEnd(0)

In [43]:
# Смотрим структуру данных в столбце Has_Chip и количество уникальных значений
data['Has_Chip'].value_counts()

Has_Chip
YES    21910227
NO      2476673
Name: count, dtype: int64

In [44]:
# Кодируем наличие чипа в карте как число, где 1 - есть чип, 0 - нет чипа и переводим в int8 для оптимизации памяти
data['Has_Chip'] = data['Has_Chip'].map({'YES': 1, 'NO': 0}).astype('int8')

In [45]:
data.head()

,User,Card,Timestamp,Amount,Use_Chip,Merchant_ID,Merchant_City,Merchant_State,Merchant_Zip,MCC,Errors,Fraud,Person,Current_Age,Retirement_Age,Birth_Year,Birth_Month,Gender,User_Address,User_Apartment,User_City,User_State,User_Zip,User_Latitude,User_Longitude,Per_Capita_Income_Zipcode,Yearly_Income,Total_Debt,FICO,Num_Credit_Cards,Card_Index,Card_Brand,Card_Type,Card_Number,Expires,CVV,Has_Chip,Cards_Issued,Credit_Limit,Account_Open_Date,PIN_Last_Changed_Year,Card_on_Dark_Web
0,0,0,2002-09-01 06:21:00,134.09,Swipe Transaction,69374,La Verne,CA,91750.00,5300,NaN,No,Hazel Robinson,53,66,1966,11,0,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.00,59696.00,127613.00,787,5,0,Visa,Debit,4344676511950444,2022-12-31,623,1,2,$24295,09/2002,2008,No
1,0,0,2002-09-01 06:42:00,38.48,Swipe Transaction,46284,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,0,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.00,59696.00,127613.00,787,5,0,Visa,Debit,4344676511950444,2022-12-31,623,1,2,$24295,09/2002,2008,No
2,0,0,2002-09-02 06:22:00,120.34,Swipe Transaction,46284,Monterey Park,CA,91754.00,5411,NaN,No,Hazel Robinson,53,66,1966,11,0,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.00,59696.00,127613.00,787,5,0,Visa,Debit,4344676511950444,2022-12-31,623,1,2,$24295,09/2002,2008,No
3,0,0,2002-09-02 17:45:00,128.95,Swipe Transaction,68751,Monterey Park,CA,91754.00,5651,NaN,No,Hazel Robinson,53,66,1966,11,0,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.00,59696.00,127613.00,787,5,0,Visa,Debit,4344676511950444,2022-12-31,623,1,2,$24295,09/2002,2008,No
4,0,0,2002-09-03 06:23:00,104.71,Swipe Transaction,81833,La Verne,CA,91750.00,5912,NaN,No,Hazel Robinson,53,66,1966,11,0,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,29278.00,59696.00,127613.00,787,5,0,Visa,Debit,4344676511950444,2022-12-31,623,1,2,$24295,09/2002,2008,No


In [46]:
# Переводим кредитный лимит в числовой формат
data["Credit_Limit"]=data["Credit_Limit"].str.replace("$","").astype('float32')

In [ ]:
# Переводим Account_Open_Date в формат datetime
data['Account_Open_Date'] = pd.to_datetime(data['Account_Open_Date'], format='%m/%Y', errors='coerce')

In [48]:
# Переводим PIN_Last_Changed_Year в числовой формат
data['PIN_Last_Changed_Year'] = pd.to_numeric(data['PIN_Last_Changed_Year'], errors='coerce').astype('int16')

In [49]:
# Проверяем распределение признака Card_on_Dark_Web и количество уникальных значений
print(data['Card_on_Dark_Web'].value_counts())

Card_on_Dark_Web
No    24386900
Name: count, dtype: int64


In [50]:
# Признак Card_on_Dark_Web очевидно не информативен т.к. имеет только 1 уникальное значение - NO. Удалим его из датасета
data.drop(columns=['Card_on_Dark_Web'], inplace=True)

In [51]:
# Проверяем результаты всех преобразований
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24386900 entries, 0 to 24386899
Data columns (total 41 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   User                       int16         
 1   Card                       int8          
 2   Timestamp                  datetime64[ns]
 3   Amount                     float32       
 4   Use_Chip                   category      
 5   Merchant_ID                category      
 6   Merchant_City              category      
 7   Merchant_State             category      
 8   Merchant_Zip               category      
 9   MCC                        category      
 10  Errors                     object        
 11  Fraud                      object        
 12  Person                     object        
 13  Current_Age                int8          
 14  Retirement_Age             int8          
 15  Birth_Year                 int16         
 16  Birth_Month                int8   

Мы сократили расход памяти с 8.2 гигабайта до 3.5 гигабайт, что существенно поможет нам при построении модели. Также мы удалили очевидно не информативные признаки и объеденили колонки, связанные с датами и временем. Теперь сохраним получившийся Dataset в формате Parquet который позволяет сохранить типы данных.

## 1.3 Сохраняем результат

In [52]:
data.to_parquet("fraud.parquet", index=False)

Мы сохранили полученный датасет как исходный материал для дальнейшей работы. Продолжение в ноутбуку 2.EDA
Такое разбиение поможет структурировать работу, убережт от необходимости постоянно "пересоздавать" датасет, а также убережет нас от не оптимальной работы с памятью, когда Python не освобождает оперативную память от уже не нужных данных.